# Chapter 8.6: Implementing a New Model in SGLang -- Step by Step

## Learning Objectives

By the end of this notebook, you will:

1. **Understand** SGLang's model class structure
2. **Implement** forward methods for extend vs decode mode
3. **Integrate** with FlashInfer attention
4. **Handle** weight loading and sharding
5. **Build** a complete working model
6. **Test** and validate the model

---

## Prerequisites
- Chapter 8.2 (vLLM model implementation)
- Chapter 8.5 (SGLang registration)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part8/chapter_8.6_sglang_new_model.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part8/chapter_8.6_sglang_new_model.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. SGLang Model Architecture

SGLang models follow the same decoder-only transformer pattern as vLLM, but with SGLang-specific attention integration.

```
+----------------------------------------+
|       MyModelForCausalLM (SGLang)      |
+----------------------------------------+
|                                        |
|  embed_tokens (VocabParallelEmbedding) |
|       |                                |
|  +----------------------------------+  |
|  | MyDecoderLayer (x num_layers)    |  |
|  |                                  |  |
|  |  input_layernorm (RMSNorm)       |  |
|  |       |                          |  |
|  |  MyAttention                     |  |
|  |    - qkv_proj (QKVParallel)      |  |
|  |    - o_proj (RowParallel)        |  |
|  |    - rotary_emb (RoPE)           |  |
|  |    - attn (RadixAttention) <---  |  |  SGLang specific!
|  |       |                          |  |
|  |  post_attn_layernorm (RMSNorm)   |  |
|  |       |                          |  |
|  |  MyMLP                           |  |
|  |    - gate_up_proj                |  |
|  |    - down_proj                   |  |
|  |    - act_fn (SiLU)               |  |
|  +----------------------------------+  |
|       |                                |
|  norm (RMSNorm)                        |
|       |                                |
|  lm_head (ParallelLMHead)              |
|       |                                |
|  logits_processor  <--- SGLang specific|
+----------------------------------------+
```

## 2. Step 1: Implement the MLP

The MLP is identical between vLLM and SGLang since it uses the same linear layer primitives.

In [ ]:
import torch
import torch.nn as nn
from typing import Optional, List, Tuple, Iterable, Set

# For this tutorial, we simulate vLLM's layer primitives
class SimColumnParallel(nn.Module):
    def __init__(self, in_f, out_sizes, bias=False, **kwargs):
        super().__init__()
        self.linear = nn.Linear(in_f, sum(out_sizes), bias=bias)
    def forward(self, x):
        return self.linear(x), None

class SimRowParallel(nn.Module):
    def __init__(self, in_f, out_f, bias=False, **kwargs):
        super().__init__()
        self.linear = nn.Linear(in_f, out_f, bias=bias)
    def forward(self, x):
        return self.linear(x), None

class SimQKVParallel(nn.Module):
    def __init__(self, hidden_size, head_dim, num_heads, num_kv_heads, bias=False, **kwargs):
        super().__init__()
        total = (num_heads + 2 * num_kv_heads) * head_dim
        self.linear = nn.Linear(hidden_size, total, bias=bias)
    def forward(self, x):
        return self.linear(x), None


class SGLangMLP(nn.Module):
    """
    SwiGLU MLP for SGLang.
    Identical to vLLM's implementation.
    """
    
    def __init__(self, hidden_size, intermediate_size, quant_config=None):
        super().__init__()
        self.gate_up_proj = SimColumnParallel(
            hidden_size, [intermediate_size, intermediate_size],
            bias=False, quant_config=quant_config
        )
        self.down_proj = SimRowParallel(
            intermediate_size, hidden_size,
            bias=False, quant_config=quant_config
        )
        self.act_fn = nn.SiLU()
    
    def forward(self, x):
        gate_up, _ = self.gate_up_proj(x)
        d = gate_up.shape[-1] // 2
        gate, up = gate_up[..., :d], gate_up[..., d:]
        x = self.act_fn(gate) * up
        x, _ = self.down_proj(x)
        return x

# Test
mlp = SGLangMLP(256, 512)
x = torch.randn(8, 256)
out = mlp(x)
print(f"MLP: {x.shape} -> {out.shape}")

## 3. Step 2: Implement the Attention Layer

This is where SGLang differs from vLLM. SGLang uses `RadixAttention` instead of vLLM's `Attention`.

In [ ]:
class SimRadixAttention(nn.Module):
    """
    Simulated RadixAttention for tutorial purposes.
    
    In real SGLang:
    from sglang.srt.layers.attention import RadixAttention
    
    RadixAttention handles:
    1. Storing K, V into paged KV cache
    2. Dispatching to FlashInfer for attention
    3. Extend (prefill) vs decode mode
    4. Integration with RadixCache prefix sharing
    """
    
    def __init__(self, num_heads, head_dim, scaling, num_kv_heads, layer_id):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.scaling = scaling
        self.num_kv_heads = num_kv_heads
        self.layer_id = layer_id
    
    def forward(self, q, k, v, forward_batch=None):
        # Simplified attention (real version uses FlashInfer)
        # q: [num_tokens, num_heads * head_dim]
        # k: [num_tokens, num_kv_heads * head_dim]
        # v: [num_tokens, num_kv_heads * head_dim]
        
        # In real SGLang:
        # 1. Write K, V to paged KV cache
        # 2. Call FlashInfer extend or decode kernel
        # 3. Return attention output
        
        # Simplified: just return q (no actual attention)
        return q


class SGLangAttention(nn.Module):
    """
    Attention layer for SGLang.
    
    Key difference from vLLM:
    - Uses RadixAttention instead of Attention
    - Receives forward_batch instead of kv_cache + attn_metadata
    - layer_id passed to RadixAttention for KV cache addressing
    """
    
    def __init__(
        self,
        config,
        layer_id: int,  # SGLang needs layer_id!
        quant_config=None,
    ):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = getattr(config, 'num_key_value_heads', self.num_heads)
        self.head_dim = self.hidden_size // self.num_heads
        self.scaling = self.head_dim ** -0.5
        
        # QKV projection
        self.qkv_proj = SimQKVParallel(
            self.hidden_size, self.head_dim,
            self.num_heads, self.num_kv_heads,
            bias=False, quant_config=quant_config
        )
        
        # Output projection
        self.o_proj = SimRowParallel(
            self.hidden_size, self.hidden_size,
            bias=False, quant_config=quant_config
        )
        
        # RoPE
        # In real SGLang:
        # self.rotary_emb = get_rope(self.head_dim, ...)
        
        # SGLang's RadixAttention (the key difference!)
        self.attn = SimRadixAttention(
            num_heads=self.num_heads,
            head_dim=self.head_dim,
            scaling=self.scaling,
            num_kv_heads=self.num_kv_heads,
            layer_id=layer_id,  # Layer ID for KV cache
        )
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        positions: torch.Tensor,
        forward_batch,  # ForwardBatch (SGLang-specific)
    ) -> torch.Tensor:
        # QKV projection
        qkv, _ = self.qkv_proj(hidden_states)
        
        # Split Q, K, V
        q_size = self.num_heads * self.head_dim
        kv_size = self.num_kv_heads * self.head_dim
        q = qkv[..., :q_size]
        k = qkv[..., q_size:q_size + kv_size]
        v = qkv[..., q_size + kv_size:]
        
        # Apply RoPE
        # q, k = self.rotary_emb(positions, q, k)
        
        # RadixAttention (passes forward_batch, NOT kv_cache)
        attn_output = self.attn(q, k, v, forward_batch)
        
        # Output projection
        output, _ = self.o_proj(attn_output)
        return output

# Test
class TestConfig:
    hidden_size = 256
    num_attention_heads = 4
    num_key_value_heads = 2
    intermediate_size = 512
    num_hidden_layers = 4

config = TestConfig()
attn = SGLangAttention(config, layer_id=0)
x = torch.randn(8, 256)
positions = torch.arange(8)
out = attn(x, positions, forward_batch=None)
print(f"SGLang Attention: {x.shape} -> {out.shape}")
print(f"  layer_id: {attn.attn.layer_id}")

## 4. Step 3: Implement the Decoder Layer

In [ ]:
class SGLangDecoderLayer(nn.Module):
    """
    Transformer decoder layer for SGLang.
    
    Structure is identical to vLLM, but forward() receives
    forward_batch instead of kv_cache + attn_metadata.
    """
    
    def __init__(self, config, layer_id: int, quant_config=None):
        super().__init__()
        self.hidden_size = config.hidden_size
        
        self.self_attn = SGLangAttention(config, layer_id, quant_config)
        self.mlp = SGLangMLP(config.hidden_size, config.intermediate_size, quant_config)
        
        self.input_layernorm = nn.LayerNorm(config.hidden_size)
        self.post_attention_layernorm = nn.LayerNorm(config.hidden_size)
    
    def forward(
        self,
        hidden_states: torch.Tensor,
        positions: torch.Tensor,
        forward_batch,
        residual: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        # Pre-attention norm + residual
        if residual is None:
            residual = hidden_states
            hidden_states = self.input_layernorm(hidden_states)
        else:
            hidden_states = self.input_layernorm(hidden_states)
        
        # Self attention
        hidden_states = self.self_attn(hidden_states, positions, forward_batch)
        hidden_states = residual + hidden_states
        
        # Post-attention norm + MLP
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states
        
        return hidden_states, residual

layer = SGLangDecoderLayer(config, layer_id=0)
x = torch.randn(8, 256)
out, res = layer(x, torch.arange(8), forward_batch=None)
print(f"Decoder layer: {x.shape} -> {out.shape}")

## 5. Step 4: Implement the Full Model

In [ ]:
class SGLangModel(nn.Module):
    """Transformer backbone for SGLang."""
    
    def __init__(self, config, quant_config=None):
        super().__init__()
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            SGLangDecoderLayer(config, layer_id=i, quant_config=quant_config)
            for i in range(config.num_hidden_layers)
        ])
        self.norm = nn.LayerNorm(config.hidden_size)
    
    def forward(self, input_ids, positions, forward_batch, input_embeds=None):
        if input_embeds is not None:
            hidden_states = input_embeds
        else:
            hidden_states = self.embed_tokens(input_ids)
        
        residual = None
        for layer in self.layers:
            hidden_states, residual = layer(
                hidden_states, positions, forward_batch, residual
            )
        
        hidden_states = self.norm(hidden_states)
        return hidden_states


class SGLangModelForCausalLM(nn.Module):
    """
    Complete causal LM for SGLang.
    
    This is the top-level class registered with SGLang's ModelRegistry.
    """
    
    def __init__(self, config, quant_config=None):
        super().__init__()
        self.config = config
        self.model = SGLangModel(config, quant_config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
    
    def forward(
        self,
        input_ids: torch.Tensor,
        positions: torch.Tensor,
        forward_batch,
        input_embeds: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        SGLang forward pass.
        
        Args:
            input_ids: [num_tokens] - token IDs
            positions: [num_tokens] - position IDs  
            forward_batch: ForwardBatch with attention metadata
            input_embeds: Optional pre-computed embeddings (for multimodal)
        
        Returns:
            logits: [num_tokens, vocab_size] or [batch_size, vocab_size]
        """
        hidden_states = self.model(input_ids, positions, forward_batch, input_embeds)
        
        # In real SGLang, LogitsProcessor handles:
        # 1. Only computing logits for last tokens (during extend)
        # 2. Vocabulary parallelism
        # logits = self.logits_processor(input_ids, hidden_states, 
        #                               self.lm_head.weight, forward_batch)
        
        logits = self.lm_head(hidden_states)
        return logits
    
    def load_weights(self, weights: Iterable[Tuple[str, torch.Tensor]]):
        """Load weights (same pattern as vLLM)."""
        stacked_params_mapping = [
            ("qkv_proj", "q_proj", "q"),
            ("qkv_proj", "k_proj", "k"),
            ("qkv_proj", "v_proj", "v"),
            ("gate_up_proj", "gate_proj", 0),
            ("gate_up_proj", "up_proj", 1),
        ]
        
        params_dict = dict(self.named_parameters())
        loaded = set()
        
        for name, loaded_weight in weights:
            if "rotary_emb.inv_freq" in name:
                continue
            
            is_stacked = False
            for (merged, shard, shard_id) in stacked_params_mapping:
                if shard in name:
                    merged_name = name.replace(shard, merged)
                    if merged_name in params_dict:
                        loaded.add(name)
                        is_stacked = True
                    break
            
            if not is_stacked and name in params_dict:
                params_dict[name].data.copy_(loaded_weight)
                loaded.add(name)
        
        return loaded

# Test the complete model
class FullConfig:
    vocab_size = 1000
    hidden_size = 256
    num_attention_heads = 4
    num_key_value_heads = 2
    num_hidden_layers = 4
    intermediate_size = 512

config = FullConfig()
model = SGLangModelForCausalLM(config)

input_ids = torch.randint(0, 1000, (16,))
positions = torch.arange(16)

with torch.no_grad():
    logits = model(input_ids, positions, forward_batch=None)

print(f"SGLang Model: {type(model).__name__}")
print(f"  Input: {input_ids.shape}")
print(f"  Output: {logits.shape}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Layers: {config.num_hidden_layers}")

## 6. Extend vs Decode Mode

SGLang explicitly distinguishes between extend (prefill) and decode modes.

In [ ]:
# Understanding extend vs decode in SGLang

print("Extend (Prefill) vs Decode in SGLang")
print("=" * 60)
print("""
EXTEND MODE (Prefill):
  - Processes the full input prompt
  - Multiple tokens per sequence
  - Uses FlashInfer's extend kernel
  - Stores all K, V into KV cache
  - Only needs logits for the LAST token
  
  Example:
    Batch: ["Hello world how", "What is"]
    input_ids: [Hello, world, how, What, is]  (5 tokens total)
    seq_lens: [3, 2]
    forward_mode: "extend"

DECODE MODE:
  - Generates one token at a time
  - Exactly one token per sequence
  - Uses FlashInfer's decode kernel
  - Reads from KV cache + stores new K, V
  - Needs logits for ALL tokens (each is the "last" token)
  
  Example:
    Batch: [next_token_for_seq1, next_token_for_seq2]
    input_ids: [are, the]  (2 tokens, one per sequence)
    seq_lens: [4, 3]  (total length including cached)
    forward_mode: "decode"

The model code is the SAME for both modes.
RadixAttention handles the dispatch internally.
""")

# Simulate the difference
print("\nExtend mode example:")
extend_batch = {
    "input_ids": [100, 200, 300, 400, 500],  # All prompt tokens
    "positions": [0, 1, 2, 0, 1],
    "seq_lens": [3, 2],
    "mode": "extend",
    "logits_needed_for": [2, 4],  # Last token positions only
}
for k, v in extend_batch.items():
    print(f"  {k}: {v}")

print("\nDecode mode example:")
decode_batch = {
    "input_ids": [600, 700],  # One new token per sequence
    "positions": [3, 2],  # Next positions
    "seq_lens": [4, 3],  # Total lengths (including cached)
    "mode": "decode",
    "logits_needed_for": [0, 1],  # All tokens
}
for k, v in decode_batch.items():
    print(f"  {k}: {v}")

## 7. The Full SGLang Model File

Let's see what a complete model file looks like for SGLang.

In [ ]:
COMPLETE_SGLANG_MODEL = '''
"""Complete Llama-style model for SGLang.

File: sglang/srt/models/my_llama.py
"""

import torch
import torch.nn as nn
from typing import Iterable, Optional, Set, Tuple

# Shared with vLLM
from vllm.model_executor.layers.activation import SiluAndMul
from vllm.model_executor.layers.layernorm import RMSNorm
from vllm.model_executor.layers.linear import (
    MergedColumnParallelLinear,
    QKVParallelLinear,
    RowParallelLinear,
)
from vllm.model_executor.layers.rotary_embedding import get_rope
from vllm.model_executor.layers.vocab_parallel_embedding import (
    ParallelLMHead,
    VocabParallelEmbedding,
)
from vllm.model_executor.model_loader.weight_utils import default_weight_loader

# SGLang-specific
from sglang.srt.layers.attention import RadixAttention
from sglang.srt.layers.logits_processor import LogitsProcessor


class MyLlamaMLP(nn.Module):
    def __init__(self, hidden_size, intermediate_size, quant_config=None):
        super().__init__()
        self.gate_up_proj = MergedColumnParallelLinear(
            hidden_size, [intermediate_size] * 2,
            bias=False, quant_config=quant_config,
        )
        self.down_proj = RowParallelLinear(
            intermediate_size, hidden_size,
            bias=False, quant_config=quant_config,
        )
        self.act_fn = SiluAndMul()

    def forward(self, x):
        gate_up, _ = self.gate_up_proj(x)
        x = self.act_fn(gate_up)
        x, _ = self.down_proj(x)
        return x


class MyLlamaAttention(nn.Module):
    def __init__(self, config, layer_id, quant_config=None):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = getattr(config, "num_key_value_heads", self.num_heads)

        self.qkv_proj = QKVParallelLinear(
            self.hidden_size, self.head_dim,
            self.num_heads, self.num_kv_heads,
            bias=False, quant_config=quant_config,
        )
        self.o_proj = RowParallelLinear(
            self.num_heads * self.head_dim, self.hidden_size,
            bias=False, quant_config=quant_config,
        )
        self.rotary_emb = get_rope(
            self.head_dim,
            rotary_dim=self.head_dim,
            max_position=config.max_position_embeddings,
            base=config.rope_theta,
        )
        
        # SGLang: RadixAttention with layer_id
        self.attn = RadixAttention(
            self.num_heads,
            self.head_dim,
            self.head_dim ** -0.5,
            self.num_kv_heads,
            layer_id=layer_id,
        )

    def forward(self, hidden_states, positions, forward_batch):
        qkv, _ = self.qkv_proj(hidden_states)
        q, k, v = qkv.split(
            [self.num_heads * self.head_dim,
             self.num_kv_heads * self.head_dim,
             self.num_kv_heads * self.head_dim],
            dim=-1,
        )
        q, k = self.rotary_emb(positions, q, k)
        attn_output = self.attn(q, k, v, forward_batch)
        output, _ = self.o_proj(attn_output)
        return output


class MyLlamaDecoderLayer(nn.Module):
    def __init__(self, config, layer_id, quant_config=None):
        super().__init__()
        self.self_attn = MyLlamaAttention(config, layer_id, quant_config)
        self.mlp = MyLlamaMLP(
            config.hidden_size, config.intermediate_size, quant_config,
        )
        self.input_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, hidden_states, positions, forward_batch, residual):
        if residual is None:
            residual = hidden_states
            hidden_states = self.input_layernorm(hidden_states)
        else:
            hidden_states, residual = self.input_layernorm(hidden_states, residual)
        hidden_states = self.self_attn(hidden_states, positions, forward_batch)
        hidden_states, residual = self.post_attention_layernorm(hidden_states, residual)
        hidden_states = self.mlp(hidden_states)
        return hidden_states, residual


class MyLlamaModel(nn.Module):
    def __init__(self, config, quant_config=None):
        super().__init__()
        self.embed_tokens = VocabParallelEmbedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([
            MyLlamaDecoderLayer(config, i, quant_config)
            for i in range(config.num_hidden_layers)
        ])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(self, input_ids, positions, forward_batch, input_embeds=None):
        hidden_states = input_embeds if input_embeds is not None else self.embed_tokens(input_ids)
        residual = None
        for layer in self.layers:
            hidden_states, residual = layer(hidden_states, positions, forward_batch, residual)
        hidden_states, _ = self.norm(hidden_states, residual)
        return hidden_states


class MyLlamaForCausalLM(nn.Module):
    """Top-level model registered with SGLang."""

    def __init__(self, config, quant_config=None):
        super().__init__()
        self.config = config
        self.model = MyLlamaModel(config, quant_config)
        self.lm_head = ParallelLMHead(config.vocab_size, config.hidden_size)
        self.logits_processor = LogitsProcessor(config)

    def forward(self, input_ids, positions, forward_batch, input_embeds=None):
        hidden_states = self.model(input_ids, positions, forward_batch, input_embeds)
        return self.logits_processor(
            input_ids, hidden_states, self.lm_head.weight, forward_batch
        )

    def load_weights(self, weights):
        stacked_params = [
            ("qkv_proj", "q_proj", "q"),
            ("qkv_proj", "k_proj", "k"),
            ("qkv_proj", "v_proj", "v"),
            ("gate_up_proj", "gate_proj", 0),
            ("gate_up_proj", "up_proj", 1),
        ]
        params_dict = dict(self.named_parameters())
        loaded = set()
        for name, weight in weights:
            if "rotary_emb.inv_freq" in name:
                continue
            for (p, w, s) in stacked_params:
                if w not in name:
                    continue
                name = name.replace(w, p)
                if name in params_dict:
                    param = params_dict[name]
                    weight_loader = param.weight_loader
                    weight_loader(param, weight, s)
                    loaded.add(name)
                break
            else:
                if name in params_dict:
                    param = params_dict[name]
                    wl = getattr(param, "weight_loader", default_weight_loader)
                    wl(param, weight)
                    loaded.add(name)
        return loaded
'''

print(COMPLETE_SGLANG_MODEL)

## 8. Porting from vLLM to SGLang: A Checklist

If you already have a vLLM model, porting to SGLang is straightforward.

In [ ]:
print("Porting Checklist: vLLM -> SGLang")
print("=" * 60)

checklist = [
    ("Replace Attention with RadixAttention",
     "from vllm.attention import Attention",
     "from sglang.srt.layers.attention import RadixAttention"),
    ("Add layer_id to RadixAttention constructor",
     "self.attn = Attention(num_heads, head_dim, scaling, num_kv_heads)",
     "self.attn = RadixAttention(num_heads, head_dim, scaling, num_kv_heads, layer_id=layer_id)"),
    ("Change forward() signature",
     "def forward(self, input_ids, positions, kv_caches, attn_metadata)",
     "def forward(self, input_ids, positions, forward_batch)"),
    ("Change attention call",
     "self.attn(q, k, v, kv_cache, attn_metadata)",
     "self.attn(q, k, v, forward_batch)"),
    ("Replace LogitsProcessor",
     "from vllm.model_executor.layers.logits_processor import LogitsProcessor",
     "from sglang.srt.layers.logits_processor import LogitsProcessor"),
    ("Update logits_processor call",
     "self.logits_processor(self.lm_head, hidden_states, sampling_metadata)",
     "self.logits_processor(input_ids, hidden_states, self.lm_head.weight, forward_batch)"),
    ("Keep everything else the same!",
     "Linear layers, norms, RoPE, weight loading...",
     "All shared with vLLM - no changes needed"),
]

for i, (desc, vllm_code, sglang_code) in enumerate(checklist, 1):
    print(f"\n{i}. {desc}")
    print(f"   vLLM:   {vllm_code}")
    print(f"   SGLang: {sglang_code}")

## 9. Testing and Validation

In [ ]:
# Test the model with simulated inputs

def test_sglang_model(model, config):
    """Test a SGLang model with various inputs."""
    print(f"Testing {type(model).__name__}")
    print("=" * 40)
    
    # Test 1: Basic forward pass
    print("\nTest 1: Basic forward pass")
    input_ids = torch.randint(0, config.vocab_size, (16,))
    positions = torch.arange(16)
    with torch.no_grad():
        logits = model(input_ids, positions, forward_batch=None)
    assert logits.shape[-1] == config.vocab_size, f"Wrong vocab size: {logits.shape}"
    print(f"  PASS: Output shape {logits.shape}")
    
    # Test 2: Different sequence lengths
    print("\nTest 2: Variable sequence lengths")
    for seq_len in [1, 4, 32, 128]:
        input_ids = torch.randint(0, config.vocab_size, (seq_len,))
        positions = torch.arange(seq_len)
        with torch.no_grad():
            logits = model(input_ids, positions, forward_batch=None)
        print(f"  seq_len={seq_len}: output shape {logits.shape} - PASS")
    
    # Test 3: Gradient flow
    print("\nTest 3: Gradient flow")
    input_ids = torch.randint(0, config.vocab_size, (8,))
    positions = torch.arange(8)
    logits = model(input_ids, positions, forward_batch=None)
    loss = logits.sum()
    loss.backward()
    has_grads = all(p.grad is not None for p in model.parameters() if p.requires_grad)
    print(f"  All parameters have gradients: {has_grads} - {'PASS' if has_grads else 'FAIL'}")
    model.zero_grad()
    
    # Test 4: Weight loading
    print("\nTest 4: Weight loading")
    fake_weights = [
        (name, torch.randn_like(param))
        for name, param in model.named_parameters()
    ]
    loaded = model.load_weights(fake_weights)
    print(f"  Loaded {len(loaded)} / {sum(1 for _ in model.parameters())} parameters")
    
    print("\nAll tests passed!")

# Run tests
test_sglang_model(model, FullConfig())

## 10. Exercises

### Exercise 1: Implement a GPT-2 Style Model for SGLang
Implement a GPT-2 style model with these differences:
- Uses LayerNorm instead of RMSNorm
- Uses GELU instead of SiLU
- Has bias in linear layers

### Exercise 2: Port a vLLM Model to SGLang
Take the Qwen2ForCausalLM from Chapter 8.2 and port it to SGLang. Document every change you make.

### Exercise 3: Implement Extend-Only Logits Optimization
Implement the optimization where during extend mode, logits are only computed for the last token of each sequence.

In [ ]:
# Exercise 3: Extend-only logits optimization

class OptimizedLogitsProcessor:
    """Compute logits only for needed tokens."""
    
    def __call__(self, input_ids, hidden_states, lm_head_weight, forward_batch):
        if forward_batch is not None and getattr(forward_batch, 'forward_mode', None) == 'extend':
            # Only compute logits for last token of each sequence
            last_indices = torch.cumsum(forward_batch.seq_lens, dim=0) - 1
            hidden_states = hidden_states[last_indices]
        
        logits = hidden_states @ lm_head_weight.t()
        return logits

# Demo
processor = OptimizedLogitsProcessor()
hidden = torch.randn(10, 256)  # 10 total tokens
weight = torch.randn(1000, 256)

# Full computation (decode mode)
full_logits = processor(None, hidden, weight, None)
print(f"Decode mode: {hidden.shape[0]} tokens -> {full_logits.shape} logits")

# Optimized (extend mode) 
from dataclasses import dataclass
@dataclass
class MockBatch:
    forward_mode: str
    seq_lens: torch.Tensor

batch = MockBatch(forward_mode='extend', seq_lens=torch.tensor([4, 3, 3]))
opt_logits = processor(None, hidden, weight, batch)
print(f"Extend mode: {hidden.shape[0]} tokens -> {opt_logits.shape} logits (3 sequences)")
print(f"Saved {(10-3) * 1000 * 4 / 1024:.0f} KB of computation")

## Summary

In this notebook, we implemented a complete model for SGLang:

1. **MLP**: Identical to vLLM (uses same linear layers)
2. **Attention**: Uses `RadixAttention` with `layer_id` and `forward_batch`
3. **Decoder Layer**: Same structure, different forward() signature
4. **Full Model**: `forward(input_ids, positions, forward_batch)`
5. **Weight Loading**: Same stacked_params_mapping pattern
6. **LogitsProcessor**: Only computes last-token logits during extend

### Porting Summary
To port a model from vLLM to SGLang, you only need to:
1. Replace `Attention` with `RadixAttention` (add `layer_id`)
2. Change forward() to accept `forward_batch` instead of `kv_caches + attn_metadata`
3. Update the LogitsProcessor
4. Everything else stays the same!

### Next: Chapter 8.7 -- Testing and Validating Model Correctness